# **Deep Learning Course Assignment:**# *Quadruplet network with attribute recognition and quadruplet loss*______## **Giacomo Lazzerini**## **Dario Fabiani**_____### **University of Trento**This notebook contains the implementation of our model that tries to accomplish the following tasks:* **1)   Attribute Recognition**: train the network with labelled images and predict the 29 attributes from new unlabelled images.* **2)   Person Re-Identification**: Given a list of query images, return a list of images of the same person

---> **Note on this revision.** This notebook is the *fixed* version of the original> assignment code. The model, the losses and the overall structure are unchanged;> what changed are the bugs that made the original run produce meaningless> numbers (image/label misalignment, evaluation on shuffled features, the final> model being saved with random weights, ...). Every fix is marked with a> `# FIX:` comment and the full list is in `FIXES.md`.>> It also runs **locally** (no `google.colab` dependency) and fits in **8 GB of> VRAM**.

#Upload Dataset

First you need to download the dataset.zip file and unzip it in the local storage. The data folder will contain different subfolders:* **queries**: 2.248 *64x128 px* images of pedestrians to use as queries for testing in the re-ID task.* **test**: A list of 19.679 *64x128 px* unlabelled images of pedestrians for testing the attribute recognition task.* **train**: Each 12.989 *64x128 px* image of pedestrians have an annotated ID and cam. The train and validation set will be extracted from this folder to train the model.* "**annotations_train.csv**": This csv file contains the annotations for the 27 attributes classification task for each of the 751 different identities.

In [ ]:
# FIX: works both on Colab and on a local machine (the original hard-required
# google.colab + Google Drive, so the notebook could not run anywhere else).
import os
import sys
import subprocess
import zipfile

IN_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')

# Where the extracted dataset lives (must contain train/, test/, queries/,
# annotations_train.csv). Override DATA_ROOT / DATASET_ZIP as needed.
DATA_ROOT = os.environ.get('DATA_ROOT', '/content/data' if IN_COLAB else './data')
DATASET_ZIP = os.environ.get('DATASET_ZIP', 'dataset.zip')

if not os.path.isdir(os.path.join(DATA_ROOT, 'train')):
    if IN_COLAB and not os.path.exists(DATASET_ZIP):
        from google.colab import drive
        drive.mount('/content/gdrive')
        DATASET_ZIP = '/content/gdrive/MyDrive/DEEP_LEARNING_PROJECT/dataset.zip'
    print(f'Extracting {DATASET_ZIP} -> {DATA_ROOT}')
    with zipfile.ZipFile(DATASET_ZIP) as zf:
        zf.extractall(DATA_ROOT)

for sub in ('train', 'test', 'queries'):
    n = len(os.listdir(os.path.join(DATA_ROOT, sub)))
    print(f'{sub:8s}: {n} images')
assert os.path.exists(os.path.join(DATA_ROOT, 'annotations_train.csv'))

In [ ]:
# FIX: removed CUDA_LAUNCH_BLOCKING=1 (it serialises every kernel launch and was
# roughly halving the throughput for no reason), removed the unused/Colab-only
# imports (skimage.io, google.colab.files) and the duplicated pandas import.
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

from IPython.display import display
from PIL import Image
from tqdm.auto import tqdm

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0),
          f'| {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB')
    torch.backends.cudnn.benchmark = True

# FIX: seed everything so runs are comparable (the original left the ID split,
# the quadruplet sampling and the weight init unseeded).
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

#Model Configuration

Here we create a setup dictionary to better handle configuration

In [ ]:
config = dict(
    data_root=DATA_ROOT,
    train_path=os.path.join(DATA_ROOT, 'train'),
    test_path=os.path.join(DATA_ROOT, 'test'),
    queries_path=os.path.join(DATA_ROOT, 'queries'),
    csv_file=os.path.join(DATA_ROOT, 'annotations_train.csv'),
    mAP_rank=20,
    # FIX: the source images are 64x128, so squashing them into a 224x224 square
    # destroys the aspect ratio. 256x128 (H, W) is the standard re-ID resolution
    # and is also ~35% cheaper.
    image_size=(256, 128),
    # FIX: a "batch" here is a quadruplet, i.e. 4 images through the backbone.
    # 32 quadruplets = 128 images, which fits comfortably in 8 GB with AMP.
    batch_size=32,
    epochs=30,
    num_bottleneck=256,
    num_workers=min(8, os.cpu_count() or 2),
    learning_rate=0.001,
    weight_decay=5e-4,
    momentum=0.9,
    use_amp=True,          # mixed precision: ~2x faster and ~half the VRAM
    eval_every=1,          # how often to run the (expensive) mAP evaluation
    checkpoint='best_model.pt',
)
config

# Creating File - Separated Colors

We first read the csv file and generate the label list in order to create a Pandas dataframe containing all the attribute values.

In [ ]:
csv = pd.read_csv(config['csv_file'])

# FIX: the attribute cardinalities are now taken from the annotation schema
# instead of `complete_df.nunique()`. nunique() is computed on the *data*, so an
# attribute that happens to be constant in a split silently became "binary" and
# an attribute with a missing class silently changed the head size.
ATTRIBUTE_NAMES = csv.columns[1:].tolist()                 # 27 annotated attributes
DERIVED_ATTRIBUTES = ['upmulti', 'downmulti']              # +2 derived below
ALL_ATTRIBUTES = ATTRIBUTE_NAMES + DERIVED_ATTRIBUTES

# annotations are 1-based; we shift them to 0-based. `age` is the only
# multi-class attribute (4 classes), everything else is binary.
N_CLASSES = {a: int(csv[a].max()) for a in ATTRIBUTE_NAMES}
# att_map: 1 -> single logit + BCEWithLogits, n>2 -> n logits + CrossEntropy
att_map = [1 if N_CLASSES.get(a, 2) == 2 else N_CLASSES[a] for a in ALL_ATTRIBUTES]

headers = ['image_name', 'ID', 'cam'] + ALL_ATTRIBUTES
print(f'{len(ALL_ATTRIBUTES)} attributes, att_map = {att_map}')

In [ ]:
# FIX: the original looped over 13k files doing a `csv.loc[csv['id'] == ID]`
# lookup per file (O(N*M)); a merge does the same thing in one pass.
files = sorted(f for f in os.listdir(config['train_path']) if f.endswith('.jpg'))
complete_df = pd.DataFrame({
    'image_name': files,
    'ID': [int(f.split('_')[0]) for f in files],
    'cam': [int(f.split('_')[1][1]) for f in files],
})
complete_df = complete_df.merge(csv, left_on='ID', right_on='id', how='left').drop(columns='id')
assert complete_df[ATTRIBUTE_NAMES].notna().all().all(), 'some ID has no annotation row'

# 1-based -> 0-based labels
complete_df[ATTRIBUTE_NAMES] = complete_df[ATTRIBUTE_NAMES].astype(int) - 1

# FIX: the up/down colour groups are selected by name instead of by hardcoded
# positional slices (columns[13:21] / columns[21:-1]), which only worked by
# accident and broke as soon as a column was added or reordered.
UP_COLOURS = [a for a in ATTRIBUTE_NAMES if a.startswith('up')]
DOWN_COLOURS = [a for a in ATTRIBUTE_NAMES if a.startswith('down') and a != 'down']
complete_df['upmulti'] = (complete_df[UP_COLOURS].sum(axis=1) == 0).astype(int)
complete_df['downmulti'] = (complete_df[DOWN_COLOURS].sum(axis=1) == 0).astype(int)
complete_df = complete_df[headers]

print(complete_df.shape, '| up colours:', len(UP_COLOURS), '| down colours:', len(DOWN_COLOURS))
display(complete_df.head(5))

#Creating Train and Val

In order to preserve IDs among train and validation splits and to avoid an outperforming validation, we first divide the IDs into train and validation sizes and then create the respective dataframes.

In [ ]:
# split the 751 identities between train and validation (mutually exclusive)
IDs = sorted(complete_df['ID'].unique())
print('Total IDs: ', len(IDs))

training_ids, validation_ids = train_test_split(IDs, test_size=0.28, random_state=SEED)

# FIX: reset_index(drop=True) here as well. Downstream code indexes these frames
# positionally (.iat[index, ...]), which silently mixes rows up when the index
# is still the one inherited from complete_df.
train_df = complete_df[complete_df['ID'].isin(training_ids)].reset_index(drop=True)
valid_df = complete_df[complete_df['ID'].isin(validation_ids)].reset_index(drop=True)
print(f'train: {len(train_df)} images / {len(training_ids)} IDs   '
      f'valid: {len(valid_df)} images / {len(validation_ids)} IDs')

Here we create the test and query set from the validation set. These two dataset will be used to test the *mAP*. In person reidentification *mAP* refers to the mean of the Average Precision over all queries. The AP for a query is the area under the precision-recall curve obtained from the list of predictions considering the ground truth elements as positives and the other ones as negatives.

In [ ]:
proportion_test_queries = round(len(os.listdir(config['queries_path'])) /
                                len(os.listdir(config['test_path'])), 2)

# FIX: stratify must be a label array aligned with the frame being split.
# The original passed `list(valid_df['ID'])`, which happened to work, but the
# split then produced frames whose index no longer matched their position.
validation_test_df, validation_queries_df = train_test_split(
    valid_df,
    test_size=proportion_test_queries,
    stratify=valid_df['ID'],
    random_state=SEED)

validation_test_df = validation_test_df.reset_index(drop=True)
validation_queries_df = validation_queries_df.reset_index(drop=True)

# FIX: a query is only usable if the gallery contains at least one image of the
# same identity, otherwise its AP is 0 by construction and it only adds noise
# (and `delta_recall = 1/0` would blow up in the evaluator).
gallery_ids = set(validation_test_df['ID'])
validation_queries_df = validation_queries_df[
    validation_queries_df['ID'].isin(gallery_ids)].reset_index(drop=True)

print(f'val gallery: {len(validation_test_df)}   val queries: {len(validation_queries_df)}'
      f'   (ratio {proportion_test_queries})')

# CLASS MarketDataset

Here we introduce our `MarketDataset` class. In *train* mode `__getitem__()`returns the anchor image, the labels of all the 29 attributes **of that sameimage**, and a positive (same ID) plus two negatives (different IDs) used by the***Quadruplet Loss***. In *eval* mode it returns only the image, iterating therows of its dataframe in order; in *folder* mode it iterates a directory ofunlabelled images.> **FIX — this was the single most important bug in the original notebook.**> `__getitem__` read the *image* from `sorted(os.listdir(root_dir))[index]` but> the *labels* from `dataframe.iat[index, ...]`. Since the dataframes are> **subsets** of `complete_df` in a different order, every image was paired with> the attributes and the identity of a different person, and both `train_ds` and> `valid_ds` iterated over the very same files (the whole `train/` folder), so> the train/validation split had no effect either. Whatever the network learned,> it was learning it from randomly permuted labels.

In [ ]:
class MarketDataset(Dataset):
    """mode='train'  -> (anchor, attributes, [positive, negative1, negative2])
       mode='eval'   -> image  (iterates `dataframe` in row order)
       mode='folder' -> image  (iterates sorted(os.listdir(root_dir)))"""

    def __init__(self, root_dir, dataframe=None, transform=None, mode='train'):
        assert mode in ('train', 'eval', 'folder')
        self.root_dir = root_dir
        self.transform = transform
        self.mode = mode

        if mode == 'folder':
            self.annotations = None
            # FIX: the file list is sorted here *and* reused by the submission
            # cells, so predictions and filenames can no longer drift apart.
            self.list_dir = sorted(f for f in os.listdir(root_dir) if f.endswith('.jpg'))
        else:
            self.annotations = dataframe.reset_index(drop=True)
            # FIX: the images come from the dataframe, not from the directory
            # listing, so image <-> label alignment is guaranteed.
            self.list_dir = self.annotations['image_name'].tolist()
            self.labels = self.annotations[ALL_ATTRIBUTES].to_numpy(dtype=np.int64)
            self.ids = self.annotations['ID'].to_numpy()
            # FIX: precomputed ID -> row positions. The original rebuilt the
            # positive/negative candidate lists with a full dataframe scan on
            # *every* __getitem__ call, which dominated the epoch time.
            self.id_to_positions = {}
            for pos, i in enumerate(self.ids):
                self.id_to_positions.setdefault(i, []).append(pos)
            self.all_positions = np.arange(len(self.ids))

    def __len__(self):
        return len(self.list_dir)

    def _load(self, position):
        # FIX: .convert('RGB') - a grayscale/CMYK jpg would otherwise produce a
        # tensor with the wrong number of channels and crash Normalize.
        img = Image.open(os.path.join(self.root_dir, self.list_dir[position])).convert('RGB')
        return self.transform(img) if self.transform else img

    def __getitem__(self, index):
        anchor_image = self._load(index)

        if self.mode != 'train':
            return anchor_image

        attributes = torch.from_numpy(self.labels[index])
        raw_ID = self.ids[index]

        # FIX: the anchor itself is excluded from the positive candidates (the
        # original dropped an arbitrary element with `[1:]` and could still pick
        # the anchor, making d(anchor, positive) == 0 for free).
        candidates = [p for p in self.id_to_positions[raw_ID] if p != index]
        positive = self._load(random.choice(candidates) if candidates else index)

        # FIX: two negatives of *two different* identities, both different from
        # the anchor's. The second term of the quadruplet loss pushes apart two
        # negative identities, so drawing both from the same person (which the
        # original allowed) turns that term into noise.
        n1, n2 = 0, 0
        while (self.ids[n1] == raw_ID or self.ids[n2] == raw_ID
               or self.ids[n1] == self.ids[n2]):
            n1, n2 = random.sample(range(len(self.ids)), 2)
        first_negative = self._load(n1)
        second_negative = self._load(n2)

        return anchor_image, attributes, [positive, first_negative, second_negative]

# Transformations

Here we setup the transformations to apply to the images (validation does notneed data augmentation).> **FIX.** The original train pipeline did `Resize((224, 224))` followed by> `RandomCrop(32, padding=4)`: it resized the pedestrian to 224x224 and then> threw away 98% of it, feeding the network a random 32x32 patch. The correct> idiom is to pad and crop *back to the same size*.

In [ ]:
jitter_param = 0.4
train_tfms = T.Compose([
    T.Resize(config['image_size']),
    # FIX: pad + crop back to image_size (was RandomCrop(32) -> a 32x32 patch)
    T.RandomCrop(config['image_size'], padding=8, padding_mode='reflect'),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=jitter_param, contrast=jitter_param, saturation=jitter_param),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

valid_tfms = T.Compose([
    T.Resize(config['image_size']),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Datasets

In [ ]:
# these datasets will be used for training the model
train_ds = MarketDataset(config['train_path'], train_df, train_tfms, mode='train')
valid_ds = MarketDataset(config['train_path'], valid_df, valid_tfms, mode='train')

# these datasets will be used for computing the mAP
# FIX: valid_tfms, not train_tfms - the original augmented (random crop, colour
# jitter) the gallery images while measuring retrieval performance on them.
validation_test_ds = MarketDataset(config['train_path'], validation_test_df,
                                   valid_tfms, mode='eval')
validation_queries_ds = MarketDataset(config['train_path'], validation_queries_df,
                                      valid_tfms, mode='eval')

# these datasets will be used for generating the attribute predictions and the
# person re-ID submission file
# FIX: mode='folder' + valid_tfms. The original passed `validation_test_df` as
# annotations for the test/queries folders (meaningless) and used train_tfms,
# i.e. it ran inference on randomly augmented images.
test_ds = MarketDataset(config['test_path'], transform=valid_tfms, mode='folder')
queries_ds = MarketDataset(config['queries_path'], transform=valid_tfms, mode='folder')

print(f'train {len(train_ds)} | valid {len(valid_ds)} | val-gallery {len(validation_test_ds)} '
      f'| val-queries {len(validation_queries_ds)} | test {len(test_ds)} | queries {len(queries_ds)}')

#Dataloader

In [ ]:
loader_kw = dict(num_workers=config['num_workers'],
                 pin_memory=torch.cuda.is_available(),
                 persistent_workers=config['num_workers'] > 0)

# Train
train_loader = DataLoader(train_ds, config['batch_size'], shuffle=True,
                          drop_last=True, **loader_kw)
val_loader = DataLoader(valid_ds, config['batch_size'], shuffle=False, **loader_kw)

# mAP
# FIX: shuffle=False and drop_last=False are mandatory here. The original built
# the gallery loader with shuffle=True, drop_last=True while the ground truth was
# indexed by *dataframe row position*: the features were compared against the
# labels of unrelated images (and 1 incomplete batch of gallery images was simply
# dropped), so the reported mAP was noise.
validation_test_loader = DataLoader(validation_test_ds, config['batch_size'],
                                    shuffle=False, drop_last=False, **loader_kw)
validation_queries_loader = DataLoader(validation_queries_ds, config['batch_size'],
                                       shuffle=False, drop_last=False, **loader_kw)

# Evaluation
test_loader = DataLoader(test_ds, config['batch_size'], shuffle=False, **loader_kw)
queries_loader = DataLoader(queries_ds, config['batch_size'], shuffle=False, **loader_kw)

#Our Network

#Classification Block

This class is used to build the classification block for each attribute of theimage: a bottleneck, then batchnorm, leaky ReLU, dropout, and finally the rawlogits (no `Sigmoid()`/`Softmax()`, because the losses apply the activationthemselves).> **FIX.** Dropout was applied with the *functional* API as> `F.dropout(x, p=0.5)`. `F.dropout` defaults to `training=True`, so dropout> stayed active during `net.eval()` — every validation score, every mAP and every> submitted prediction was computed with half the bottleneck randomly zeroed. The> `nn.Dropout` module respects `train()`/`eval()`.

In [ ]:
class ClassificationBlock(nn.Module):
    def __init__(self, input_dim, class_dim, num_bottleneck=config['num_bottleneck']):
        super(ClassificationBlock, self).__init__()

        self.fc = nn.Linear(input_dim, num_bottleneck)
        self.fc2 = nn.Linear(num_bottleneck, class_dim)
        self.bns = nn.BatchNorm1d(num_bottleneck)
        # FIX: module instead of F.dropout(..., training=True by default)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x):
        x = self.fc(x)
        x = self.bns(x)
        x = F.leaky_relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

#Backbone

For the backbone we choose a ResNet18, then we generate the classification layersand produce a list of raw logits for each attribute. Both raw logits andprobabilities are returned, in order to compute the loss and the accuracyrespectively.> **FIX.** The original `forward()` built `prob_pred_label` by calling every> classification head **three more times**:> `sigmoid(head(x)) if head(x).size()[1] == 1 else softmax(head(x))`.> That is 4 evaluations of each of the 29 heads per batch — and because dropout> was active, `prob_pred_label` was not even the activation of the logits used> for the loss. The probabilities are now derived from the single set of logits.

In [ ]:
class Backbone(nn.Module):
    def __init__(self, attribute_map, model_name='resnet18'):
        super(Backbone, self).__init__()
        self.model_name = model_name
        self.class_num = len(attribute_map)
        self.attribute_map = attribute_map

        # FIX: `pretrained=True` is deprecated/removed in recent torchvision.
        try:
            model_ft = getattr(models, model_name)(weights='DEFAULT')
        except TypeError:
            model_ft = getattr(models, model_name)(pretrained=True)

        if model_name.lower().startswith('resnet'):
            # FIX: read the feature dimension from the model instead of hardcoding
            # 512 (which silently broke resnet50, where it is 2048) - and read it
            # *before* replacing the fc layer.
            self.num_ftrs = model_ft.fc.in_features
            model_ft.avgpool = nn.AdaptiveAvgPool2d((1, 1))
            model_ft.fc = nn.Sequential()
            self.features = model_ft
        elif model_name.lower().startswith('densenet'):
            self.features = nn.Sequential(model_ft.features,
                                          nn.AdaptiveAvgPool2d((1, 1)))
            self.num_ftrs = model_ft.classifier.in_features
        else:
            raise NotImplementedError(model_name)

        self.classifiers = nn.ModuleList([
            ClassificationBlock(input_dim=self.num_ftrs, class_dim=c) for c in attribute_map])

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)

        # FIX: heads evaluated once; probabilities derived from those logits.
        raw_pred_label = [head(x) for head in self.classifiers]
        prob_pred_label = [torch.sigmoid(o) if o.size(1) == 1 else torch.softmax(o, dim=1)
                           for o in raw_pred_label]

        return raw_pred_label, prob_pred_label, x

#Loss FunctionsIn this work, we implemented two loss:* **Attribute Loss** : here we sum all the attribute losses. If the class is binary we compute a Sigmoid layer and then the Binary Crossentropy using ***nn.BCEWithLogitsLoss()***, if is multiclass we compute a Softmax layer to the Cross-entropy loss using ***nn.CrossEntropyLoss()**** **Quadruplet Loss** : This loss is a special case of *Triplet Loss*The triplet loss is normally trained on a series of triplets $\left\{x_{a}, x_{p}, x_{n}\right\}$, where $x_{a}$ and $x_{p}$ are images from the same person, and $x_{n}$ is from a different person. The triplet loss is designed to keep $x_{a}$ closer to $x_{p}$ than $x_{n}$. It is formulated as following:**Triplet Loss:**$$L_{t r p}=\sum_{a, p, n}^{N}\left[\left\|f\left(x_{a}\right)-f\left(x_{p}\right)\right\|_{2}^{2}-\left\|f\left(x_{a}\right)-f\left(x_{n}\right)\right\|_{2}^{2}+\alpha_{t r p}\right]_{+}$$where $[z]_{+}=\max (z, 0)$, and $f\left(x_{a}\right), f\left(x_{p}\right), f\left(x_{n}\right)$ meanfeatures of three input images (anchor, positive and negative).Triplet loss trains the model only based on the relative distances between positive and negative pairs with respect to the anchor, the Quadruplet Loss introduces a new constraint which pushes away negative pairs from positive pairs. To do this anther negative image $x_{n_{2}}$ is used to compute the loss on the quadruplet: $\left\{x_{a}, x_{p}, x_{n_{1}}, x_{n_{2}}\right\}$.**Quadruplet Loss:**$\begin{aligned} L_{\text {quad }}=& \sum_{a, p, n_{1}, n_{2}}^{N}\left[\left\|f\left(x_{a}\right)-f\left(x_{p}\right)\right\|_{2}^{2}-\left\|f\left(x_{a}\right)-f\left(x_{n_{1}}\right)\right\|_{2}^{2}+\alpha_{1}\right]_{+} \\ &+\sum_{a, p, n_{1}, n_{2}}^{N}\left[\left\|f\left(x_{a}\right)-f\left(x_{p}\right)\right\|_{2}^{2}-\left\|f\left(x_{n_{1}}\right)-f\left(x_{n_{2}}\right)\right\|_{2}^{2}+\alpha_{2}\right]_{+} \\ & s_{a}=s_{p}, s_{a} \neq s_{n_{1}}, s_{a} \neq s_{n_{2}}, s_{n_{1}} \neq s_{n_{2}} \end{aligned}$where $\alpha_{1}$ and $\alpha_{2}$ are the values of margins in two terms and $s_{i}$ refers to the person ID of image $x_{a}$.

In [ ]:
class AttributesLoss(nn.Module):
    def __init__(self):
        super(AttributesLoss, self).__init__()
        # FIX: the criteria were re-instantiated inside forward() on every batch.
        self.bce = nn.BCEWithLogitsLoss()
        self.crossEntropy = nn.CrossEntropyLoss()

    def forward(self, preds, attrs):
        binary_losses = 0.
        cross_losses = 0.
        for idx, logits in enumerate(preds[0]):
            if logits.size(1) == 1:
                binary_losses = binary_losses + self.bce(
                    logits, attrs[:, idx].unsqueeze(1).to(torch.float32))
            else:
                cross_losses = cross_losses + self.crossEntropy(logits, attrs[:, idx])
        # FIX: mean over the heads, not sum. Summing 29 losses of ~0.6 each gives
        # ~17, so with lambda=0.8 the attribute term contributed ~14 against the
        # ~0.2 of the quadruplet term: the metric-learning signal - the whole point
        # of the architecture - was numerically irrelevant, roughly 70x smaller.
        # The mean also makes lambda interpretable and independent of how many
        # attributes there happen to be.
        return (cross_losses + binary_losses) / len(preds[0])


class QuadrupletLoss(torch.nn.Module):
    """Quadruplet loss function.
    Builds on the Triplet Loss and takes 4 inputs: one anchor, one positive and
    two negative examples. The negative examples need not match the anchor, the
    positive, or each other.
    """

    def __init__(self, margin1=2.0, margin2=1.0, normalize=True):
        super(QuadrupletLoss, self).__init__()
        self.margin1 = margin1
        self.margin2 = margin2
        # FIX: retrieval ranks by *cosine* similarity, while the loss pushed raw
        # unnormalised 512-d features around with fixed Euclidean margins - the
        # two disagreed. L2-normalising the embeddings makes the squared
        # Euclidean distance a monotone function of the cosine distance, so the
        # loss now optimises the quantity that is actually used at test time
        # (and bounds the distances in [0, 4], which makes the margins meaningful).
        self.normalize = normalize

    def calc_euclidean(self, x1, x2):
        return (x1 - x2).pow(2).sum(1)

    def forward(self, anchor, positive, negative1, negative2):
        if self.normalize:
            anchor, positive, negative1, negative2 = (
                F.normalize(t, dim=1) for t in (anchor, positive, negative1, negative2))

        squarred_distance_pos = self.calc_euclidean(anchor, positive)
        squarred_distance_neg = self.calc_euclidean(anchor, negative1)
        squarred_distance_neg_b = self.calc_euclidean(negative1, negative2)

        quadruplet_loss = (
            F.relu(self.margin1 + squarred_distance_pos - squarred_distance_neg)
            + F.relu(self.margin2 + squarred_distance_pos - squarred_distance_neg_b))

        return quadruplet_loss.mean()


class OverallWrapper(nn.Module):
    def __init__(self, lambda_=0.8):
        super(OverallWrapper, self).__init__()
        self.quadruplet_loss = QuadrupletLoss()
        self.attr_loss = AttributesLoss()
        self.lambda_ = lambda_

    def forward(self, preds, attrs, anchor, positive, first_negative, second_negative):
        return (self.lambda_ * self.attr_loss(preds, attrs)
                + (1 - self.lambda_) * self.quadruplet_loss(
                    anchor, positive, first_negative, second_negative))

In [ ]:
def get_cost_function():
    return OverallWrapper()

#Optimizer

In [ ]:
def get_optimizer(net, lr, wd, momentum):
    # FIX: the original was `SGD(net.parameters(), lr, momentum)` while being
    # called as `get_optimizer(net, lr, weight_decay, momentum)`. The third
    # positional argument of SGD is `momentum`, so it received weight_decay
    # (1e-6) as the momentum and the weight decay was silently never applied.
    return torch.optim.SGD(net.parameters(), lr=lr, momentum=momentum,
                           weight_decay=wd, nesterov=True)

# Metrics helper> **FIX.** Accuracy was accumulated differently in `train()` and `test()` and> both were wrong:> * `train()` did `accuracy[i] += predicted[i][1].eq(targets.transpose(0,1)[i]).sum()`,>   i.e. it compared **the prediction of the 2nd image of the batch** against the>   whole column of targets (broadcast), producing a number between 0 and>   `batch_size` that has nothing to do with accuracy;> * the `accuracy_tot` loop was nested *inside* the per-attribute loop and reused>   `i` as its own loop variable, shadowing the outer index;> * the running loss was divided by the number of *samples* although>   `loss.item()` is already a per-batch mean, so the reported loss was the real>   one divided by the batch size.

In [ ]:
class AttributeMeter:
    """Accumulates per-attribute correct counts and the running loss."""

    def __init__(self, attribute_names):
        self.names = attribute_names
        self.correct = np.zeros(len(attribute_names), dtype=np.int64)
        self.samples = 0
        self.loss_sum = 0.
        self.batches = 0

    def update(self, probs, targets, loss):
        bs = targets.size(0)
        self.samples += bs
        self.loss_sum += float(loss) * bs   # FIX: weight by batch size...
        self.batches += bs
        for i, p in enumerate(probs):
            pred = (p > 0.5).long().flatten() if p.size(1) == 1 else p.argmax(dim=1)
            self.correct[i] += int(pred.eq(targets[:, i]).sum())

    @property
    def loss(self):
        return self.loss_sum / max(self.batches, 1)   # ...and normalise by samples

    @property
    def accuracy(self):
        return (self.correct / max(self.samples, 1) * 100).tolist()

    @property
    def total_accuracy(self):
        acc = self.accuracy
        return sum(acc) / len(acc)

#Train

> **FIX.** The original ran **four separate forward passes** per batch (anchor,> positive, negative1, negative2), each keeping its own autograd graph. The four> branches share the same weights, so concatenating them into a single forward> pass gives identical gradients with one graph instead of four — roughly 4x less> Python/kernel-launch overhead and a much better GPU utilisation. Mixed> precision (AMP) is also enabled, which halves the activation memory.

In [ ]:
def _amp_ctx(enabled):
    try:
        return torch.amp.autocast('cuda', enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(enabled=enabled)


def _grad_scaler(enabled):
    try:
        return torch.amp.GradScaler('cuda', enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def forward_quadruplet(net, inputs, quadruplet, device):
    """One forward pass for the 4 branches (FIX: was 4 separate passes)."""
    bs = inputs.size(0)
    batch = torch.cat([inputs] + [q.to(device, non_blocking=True) for q in quadruplet], dim=0)
    raw, prob, feats = net(batch)
    # keep only the anchor slice for the attribute heads
    raw = [o[:bs] for o in raw]
    prob = [o[:bs] for o in prob]
    anchor, positive, neg1, neg2 = feats.split(bs, dim=0)
    return (raw, prob), anchor, positive, neg1, neg2


def train(net, data_loader, optimizer, cost_function, scaler=None, device=device):
    meter = AttributeMeter(ALL_ATTRIBUTES)
    use_amp = scaler is not None and scaler.is_enabled()

    net.train()
    for inputs, targets, quadruplet in tqdm(data_loader, desc='train', leave=False):
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        # FIX: zero_grad *before* the backward pass. The original called
        # backward() -> step() -> zero_grad(), which works only because the order
        # happens to be cyclic; with any early `continue`/exception the gradients
        # of two batches got summed. set_to_none is also cheaper.
        optimizer.zero_grad(set_to_none=True)

        with _amp_ctx(use_amp):
            outputs, anchor, positive, neg1, neg2 = forward_quadruplet(
                net, inputs, quadruplet, device)
            loss = cost_function(outputs, targets, anchor, positive, neg1, neg2)

        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        meter.update([p.detach().float() for p in outputs[1]], targets, loss.item())

    return meter.loss, meter.accuracy, meter.total_accuracy


@torch.no_grad()
def test(net, data_loader, cost_function, device=device, use_amp=False):
    """FIX: the mAP evaluation is no longer buried inside test(). The original
    recomputed the full gallery+query features every time test() was called -
    including on the *training* loader - which more than doubled the cost of an
    epoch for a number that was thrown away."""
    meter = AttributeMeter(ALL_ATTRIBUTES)

    net.eval()
    for inputs, targets, quadruplet in tqdm(data_loader, desc='eval', leave=False):
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with _amp_ctx(use_amp):
            outputs, anchor, positive, neg1, neg2 = forward_quadruplet(
                net, inputs, quadruplet, device)
            loss = cost_function(outputs, targets, anchor, positive, neg1, neg2)

        meter.update([p.float() for p in outputs[1]], targets, loss.item())

    return meter.loss, meter.accuracy, meter.total_accuracy

# Functions for Re-identification test and evaluation:Here you can find all the function used for the Re-identification task. 

In [ ]:
def get_ground_truth(gallery_df, queries_df):
    """For each query *position* returns the set of gallery *positions* with the
    same identity.

    FIX: the original compared `t['ID'] == q['ID']` over `iterrows()` and used
    `idx_t`, the pandas index. That only coincides with the position the model
    predicts when the frame has been reset_index()'d *and* the gallery loader
    preserves order - neither of which was true. This version is O(N+M) and
    works on positions explicitly.
    """
    by_id = {}
    for pos, i in enumerate(gallery_df['ID'].to_numpy()):
        by_id.setdefault(i, set()).add(pos)
    return {q_pos: by_id.get(i, set())
            for q_pos, i in enumerate(queries_df['ID'].to_numpy())}

In [ ]:
@torch.no_grad()
def image_feature(net, dataloader, device=device, use_amp=False):
    """Extract the (L2-normalised) embedding of every image in the loader.

    FIX: the original had no `torch.no_grad()` on image_feature itself - it
    relied on the caller - and it kept every batch of features on the GPU, so
    extracting 19,679 test features allocated the whole matrix in VRAM.
    """
    net.eval()
    features = []
    for inputs in tqdm(dataloader, desc='features', leave=False):
        inputs = inputs.to(device, non_blocking=True)
        with _amp_ctx(use_amp):
            _, _, f = net(inputs)
        # FIX: normalise once here, so cosine similarity is a plain dot product
        features.append(F.normalize(f.float(), dim=1).cpu())
    return torch.cat(features, dim=0)


@torch.no_grad()
def get_topk_images(model, gallery_loader, queries_loader, k=None):
    """Return the k most similar gallery images for each query image."""
    k = k or config['mAP_rank']
    gallery_features = image_feature(model, gallery_loader)
    query_features = image_feature(model, queries_loader)

    # FIX: the original allocated a (n_queries x n_gallery) matrix and filled it
    # with a python loop over the queries (2248 iterations of a 19679-wide
    # cosine_similarity). With normalised features this is one matmul; it is also
    # chunked so the 2248x19679 float matrix never has to exist all at once.
    tops = []
    chunk = 512
    for start in range(0, query_features.size(0), chunk):
        sims = query_features[start:start + chunk] @ gallery_features.t()
        tops.append(sims.topk(min(k, sims.size(1)), dim=1).indices)
    return torch.cat(tops, dim=0)


def test_mAP(model, gallery_loader, queries_loader, ground_truth_dict, k=None):
    top_k = get_topk_images(model, gallery_loader, queries_loader, k=k)
    predictions_dict = {idx: r for idx, r in enumerate(top_k.tolist())}
    return Evaluator.evaluate_map(predictions_dict, ground_truth_dict)

In [ ]:
from typing import Dict, List, Set


class Evaluator:

    @staticmethod
    def evaluate_map(predictions: Dict[str, List], ground_truth: Dict[str, Set]):
        '''
        Computes the mAP of the predictions with respect to the given ground truth.
        In person re-identification mAP refers to the mean of the AP over all queries.
        The AP for a query is the area under the precision-recall curve obtained from
        the list of predictions considering the ground truth elements as positives and
        the other ones as negatives.

        :param predictions: dictionary from query to list of gallery entries associated
                            with the query, ordered from the most to the least confident.
        :param ground_truth: dictionary from query to set of gallery entries associated
                             with the query.
        '''

        m_ap = 0.0
        for current_ground_truth_query, current_ground_truth_query_set in ground_truth.items():

            # No predictions were performed for the current query, AP = 0
            if current_ground_truth_query not in predictions:
                continue
            # FIX: a query with an empty ground truth set used to raise
            # ZeroDivisionError on `1.0 / len(...)`.
            if not current_ground_truth_query_set:
                continue

            current_ap = 0.0
            current_predictions_list = predictions[current_ground_truth_query]
            delta_recall = 1.0 / len(current_ground_truth_query_set)

            encountered_positives = 0
            for idx, current_prediction in enumerate(current_predictions_list):
                if current_prediction in current_ground_truth_query_set:
                    encountered_positives += 1
                    current_precision = encountered_positives / (idx + 1)
                    current_ap += current_precision * delta_recall

            m_ap += current_ap

        return m_ap / max(len(ground_truth), 1)


if __name__ == '__main__':
    predictions = {'a': ['a', 'bb', 'c', 'l'], 'f': ['f', 'gg', 'h']}
    ground_truth = {'a': {'a', 'b', 'c'}, 'f': {'f', 'g', 'h'}}
    print(Evaluator.evaluate_map(predictions, ground_truth))

#Main

In [ ]:
def log_values(writer, step, loss, accuracy, total_accuracy, prefix, att):
    writer.add_scalar(f'{prefix}/loss', loss, step)
    # FIX: the step was missing, so every epoch overwrote the same point.
    writer.add_scalar(f'{prefix}/Total accuracy', total_accuracy, step)
    for i in range(len(accuracy)):
        writer.add_scalar(f'{prefix}/{att[i]} accuracy', accuracy[i], step)


def print_accuracies(name, accuracy, total_accuracy, names=None):
    names = names or ALL_ATTRIBUTES
    line = '  '.join(f'{names[i]}={accuracy[i]:.1f}' for i in range(len(accuracy)))
    print(f'  {name}: {line}')
    print(f'  {name} mean accuracy: {total_accuracy:.2f}')


def main(att_map, gt_dict, validation_test_loader, validation_queries_loader,
         device=device, learning_rate=config['learning_rate'],
         weight_decay=config['weight_decay'], momentum=config['momentum'],
         epochs=config['epochs'], save_model=True):

    from torch.utils.tensorboard import SummaryWriter
    writer = SummaryWriter(log_dir='runs/exp1')

    net = Backbone(attribute_map=att_map).to(device)
    optimizer = get_optimizer(net, learning_rate, weight_decay, momentum)
    cost_function = get_cost_function()
    use_amp = config['use_amp'] and torch.cuda.is_available()
    scaler = _grad_scaler(use_amp)

    # FIX: the original evaluated the *training* loader (and its mAP) before and
    # after training, i.e. 4 extra full-dataset passes for numbers nobody used.
    val_loss, val_accuracy, val_total_accuracy = test(net, val_loader, cost_function,
                                                      device, use_amp)
    mAP = test_mAP(net, validation_test_loader, validation_queries_loader, gt_dict)
    print(f'Before training: val loss {val_loss:.5f} | mean acc {val_total_accuracy:.2f} '
          f'| mAP {mAP:.5f}')
    writer.add_scalar('mAP', mAP, -1)

    best_mAP = -1.
    for e in range(epochs):
        train_loss, train_accuracy, train_total_accuracy = train(
            net, train_loader, optimizer, cost_function, scaler, device)
        val_loss, val_accuracy, val_total_accuracy = test(
            net, val_loader, cost_function, device, use_amp)

        log_values(writer, e, train_loss, train_accuracy, train_total_accuracy,
                   'Train', ALL_ATTRIBUTES)
        log_values(writer, e, val_loss, val_accuracy, val_total_accuracy,
                   'Validation', ALL_ATTRIBUTES)

        if (e + 1) % config['eval_every'] == 0 or e == epochs - 1:
            mAP = test_mAP(net, validation_test_loader, validation_queries_loader, gt_dict)
            writer.add_scalar('mAP', mAP, e)

            # FIX: keep the *best* weights, and keep the weights of the network we
            # actually trained. The original did
            #     model = Backbone(attribute_map=att_map)
            #     torch.save(model.state_dict(), 'model')
            # i.e. it instantiated a brand new network and saved its **random**
            # initialisation. Every submitted prediction of both tasks came from
            # an untrained model.
            if save_model and mAP > best_mAP:
                best_mAP = mAP
                torch.save({'model': net.state_dict(), 'att_map': att_map,
                            'epoch': e, 'mAP': mAP, 'config': config},
                           config['checkpoint'])

        print(f'Epoch {e + 1}/{epochs} | train loss {train_loss:.5f} '
              f'mean acc {train_total_accuracy:.2f} | val loss {val_loss:.5f} '
              f'mean acc {val_total_accuracy:.2f} | mAP {mAP:.5f} (best {best_mAP:.5f})')

    print_accuracies('train', train_accuracy, train_total_accuracy)
    print_accuracies('val', val_accuracy, val_total_accuracy)
    print(f'Best mAP: {best_mAP:.5f} -> {config["checkpoint"]}')

    writer.close()
    return net

In [ ]:
ground_truth_dict = get_ground_truth(validation_test_df, validation_queries_df)
print(f'{len(ground_truth_dict)} queries, '
      f'{np.mean([len(v) for v in ground_truth_dict.values()]):.1f} relevant gallery '
      f'images per query on average')

#Run

In [ ]:
net = main(att_map=att_map,
           gt_dict=ground_truth_dict,
           validation_test_loader=validation_test_loader,
           validation_queries_loader=validation_queries_loader)

# Tensorboard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=runs

In [ ]:
# FIX: load the *trained* best checkpoint. The original overwrote the file with
# a freshly initialised network before loading it back, so `model` was random.
ckpt = torch.load(config['checkpoint'], map_location=device, weights_only=False)
model = Backbone(attribute_map=ckpt['att_map'])
model.load_state_dict(ckpt['model'])
model.to(device).eval()
print(f"loaded epoch {ckpt['epoch']} | val mAP {ckpt['mAP']:.5f}")

# Task 1: ClassificationIn this task is required to produce predictions for each image in the ***test*** directory. Here we first introduce a function ***attribute_prediction()*** that computes the attribute predictions for each test image. Then we produce a classification test.csv using a Pandas DF with all the predictions.

In [ ]:
@torch.no_grad()
def attribute_prediction(model, loader, device=device):
    """Extract the attribute predictions for every image in the data loader."""
    model.eval()
    all_predictions = []
    for images in tqdm(loader, desc='predict', leave=False):
        images = images.to(device, non_blocking=True)
        _, attributes, _ = model(images)
        predictions = []
        for output in attributes:
            if output.size(1) == 1:
                # FIX: `torch.round` on a probability is a threshold at 0.5, which
                # is what we want, but it returned a float column; .long() keeps
                # the CSV integral like the multi-class columns.
                pred = (output.squeeze(1) > 0.5).long()
            else:
                pred = torch.argmax(output, dim=1)
            predictions.append(pred.unsqueeze(1))
        all_predictions.append(torch.cat(predictions, dim=1).cpu())
    return torch.cat(all_predictions, dim=0).tolist()

In [ ]:
attributes = attribute_prediction(model, test_loader)
print(len(attributes), 'x', len(attributes[0]))

In [ ]:
attribute_recognition_file = 'classification_test.csv'

# FIX: the index used to be `os.listdir(config['test_path'])` while the features
# were produced from `sorted(os.listdir(...))`. os.listdir returns an arbitrary
# order, so every row of the submitted csv carried the predictions of a
# different image. Reusing the dataset's own (sorted) file list makes them
# aligned by construction.
attr_dataframe = pd.DataFrame(data=attributes,
                              index=test_ds.list_dir,
                              columns=ALL_ATTRIBUTES)
attr_dataframe.index.name = 'image_name'
attr_dataframe.to_csv(attribute_recognition_file)
display(attr_dataframe.head())

In [ ]:
# FIX: guarded, so the notebook does not crash outside Colab.
if IN_COLAB:
    from google.colab import files
    files.download(attribute_recognition_file)

#Task 2: Re-identification The mAP has been evaluated during train/validation on each epoch using the validation set. Here we generate the submission file with the top_k test images for each query. To compute the first top-k results we use the ***get_topk_images()*** function

In [ ]:
final_top_k = get_topk_images(model, test_loader, queries_loader)
print(tuple(final_top_k.shape))

In [ ]:
reid_file = 'reid_test.txt'
lines = [queries_ds.list_dir[idx] + ': ' +
         ', '.join(test_ds.list_dir[i] for i in ids) + '\n'
         for idx, ids in enumerate(final_top_k.tolist())]

# FIX: open in 'w' mode. The original opened 'reid_test.txt' in append mode, so
# re-running the cell kept stacking duplicated blocks of results into the
# submission file.
with open(reid_file, 'w') as f:
    f.writelines(lines)
print(''.join(lines[:2]))

if IN_COLAB:
    from google.colab import files
    files.download(reid_file)

# Visualization

In [ ]:
class_test = pd.read_csv('classification_test.csv', index_col=0)

for _ in range(3):
    # FIX: randint(0, n) can return n -> IndexError. randrange(n) cannot.
    img_idx = random.randrange(class_test.shape[0])
    im = Image.open(os.path.join(config['test_path'], class_test.index[img_idx]))
    plt.imshow(im)
    plt.axis('off')
    plt.title('Test Image')
    plt.show()
    print(class_test.iloc[img_idx])

In [ ]:
for _ in range(3):
    img_idx = random.randrange(len(queries_ds.list_dir))
    im = Image.open(os.path.join(config['queries_path'], queries_ds.list_dir[img_idx]))
    plt.imshow(im)
    plt.axis('off')
    plt.title('Query Image')
    plt.show()

    results = [test_ds.list_dir[i] for i in final_top_k[img_idx].tolist()]
    imgs = [Image.open(os.path.join(config['test_path'], r)) for r in results]
    # FIX: the grid was hardcoded to 15 columns while mAP_rank is 20, so the
    # last 5 retrieved images were silently never drawn.
    _, axs = plt.subplots(1, len(imgs), figsize=(len(imgs), 3))
    for img, ax in zip(imgs, np.atleast_1d(axs).ravel()):
        ax.imshow(img)
        ax.axis('off')
    plt.show()